In [1]:
!pip install nibabel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.6 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# === 1. Imports & Setup ===
import os
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:

# === 2. Data Loader & Segmentation Splitting ===
def load_nifti_image(filepath):
    nii = nib.load(filepath)
    return nii.get_fdata()

def split_regions(mask):
    # mask: 1=tibia, 2=femur, 0=background
    tibia = (mask == 1).astype(np.float32)
    femur = (mask == 2).astype(np.float32)
    background = (mask == 0).astype(np.float32)
    return tibia, femur, background

def preprocess_region(region):
    # (D, H, W) -> (1, 3, D, H, W)
    region = torch.tensor(region, dtype=torch.float32)
    region = region.unsqueeze(0)  # [1, D, H, W]
    region = region.repeat(3, 1, 1, 1)  # [3, D, H, W]
    region = region.unsqueeze(0)  # [1, 3, D, H, W]
    return region.to(device)

In [6]:
# === 3. 2D -> 3D DenseNet Inflation (Conv, BatchNorm, AvgPool) ===
def inflate_densenet2d_to_3d(model_2d, time_dim=3):
    def inflate_module(module):
        for name, child in list(module.named_children()):
            # Conv2d -> Conv3d
            if isinstance(child, nn.Conv2d):
                new_conv = nn.Conv3d(
                    child.in_channels,
                    child.out_channels,
                    kernel_size=(time_dim, child.kernel_size[0], child.kernel_size[1]),
                    stride=(1, child.stride[0], child.stride[1]),
                    padding=(1, child.padding[0], child.padding[1]),
                    bias=(child.bias is not None)
                )
                with torch.no_grad():
                    w2d = child.weight.data
                    w3d = w2d.unsqueeze(2).repeat(1, 1, time_dim, 1, 1) / time_dim
                    new_conv.weight.data = w3d
                    if child.bias is not None:
                        new_conv.bias.data = child.bias.data
                setattr(module, name, new_conv)
            # BatchNorm2d -> BatchNorm3d
            elif isinstance(child, nn.BatchNorm2d):
                new_bn = nn.BatchNorm3d(
                    child.num_features,
                    eps=child.eps,
                    momentum=child.momentum,
                    affine=child.affine,
                    track_running_stats=child.track_running_stats
                )
                if child.affine:
                    new_bn.weight.data = child.weight.data.clone()
                    new_bn.bias.data = child.bias.data.clone()
                new_bn.running_mean = child.running_mean.clone()
                new_bn.running_var = child.running_var.clone()
                setattr(module, name, new_bn)
            # AvgPool2d -> AvgPool3d
            elif isinstance(child, nn.AvgPool2d):
                k = child.kernel_size if isinstance(child.kernel_size, tuple) else (child.kernel_size, child.kernel_size)
                s = child.stride if isinstance(child.stride, tuple) else (child.stride, child.stride)
                p = child.padding if isinstance(child.padding, tuple) else (child.padding, child.padding)
                new_pool = nn.AvgPool3d(
                    kernel_size=(1, *k),
                    stride=(1, *s),
                    padding=(0, *p),
                    ceil_mode=child.ceil_mode,
                    count_include_pad=child.count_include_pad
                )
                setattr(module, name, new_pool)
            # MaxPool2d -> MaxPool3d
            elif isinstance(child, nn.MaxPool2d):
                k = child.kernel_size if isinstance(child.kernel_size, tuple) else (child.kernel_size, child.kernel_size)
                s = child.stride if isinstance(child.stride, tuple) else (child.stride, child.stride)
                p = child.padding if isinstance(child.padding, tuple) else (child.padding, child.padding)
                new_pool = nn.MaxPool3d(
                    kernel_size=(1, *k),
                    stride=(1, *s),
                    padding=(0, *p),
                    dilation=(1, 1, 1),
                    ceil_mode=child.ceil_mode
                )
                setattr(module, name, new_pool)
            else:
                inflate_module(child)
        return module
    model_3d = inflate_module(model_2d)
    return model_3d

def get_3d_densenet():
    model2d = models.densenet121(weights='IMAGENET1K_V1')
    model3d = inflate_densenet2d_to_3d(model2d)
    return model3d

In [12]:

# === 4. Feature Extraction with Layer Hooks ===
class FeatureExtractor3D(nn.Module):
    def __init__(self, model_3d, layer_names):
        super().__init__()
        self.model = model_3d
        self.layer_names = layer_names
        self.features = {}
        self._register_hooks()

    def _get_layer_by_name(self, name):
        parts = name.split('.')
        layer = self.model
        for p in parts:
            if p.isdigit():
                layer = layer[int(p)]
            else:
                layer = getattr(layer, p)
        return layer

    def _register_hooks(self):
        for name in self.layer_names:
            layer = self._get_layer_by_name(name)
            layer.register_forward_hook(self.save_output(name))

    def save_output(self, name):
        def hook(module, input, output):
            self.features[name] = output
        return hook

    def forward(self, x):
        self.features = {}
class FeatureExtractor3D(nn.Module):
    def __init__(self, model_3d, layer_names):
        super().__init__()
        self.model = model_3d
        self.layer_names = layer_names
        self.features = {}
        self._register_hooks()

    def _get_layer_by_name(self, name):
        parts = name.split('.')
        layer = self.model
        for p in parts:
            if p.isdigit():
                layer = layer[int(p)]
            else:
                layer = getattr(layer, p)
        return layer

    def _register_hooks(self):
        for name in self.layer_names:
            layer = self._get_layer_by_name(name)
            layer.register_forward_hook(self.save_output(name))

    def save_output(self, name):
        def hook(module, input, output):
            self.features[name] = output
        return hook

    def forward(self, x):
        self.features = {}
        _ = self.model.features(x)   # <----- Only run up to 'features'
        return self.features
        return self.features

def gap_3d(x):
    return F.adaptive_avg_pool3d(x, 1).flatten(1)

def compute_cosine(vec1, vec2):
    return F.cosine_similarity(vec1, vec2).item()


In [13]:

# === 5. Main Pipeline ===
nii_path = '/kaggle/input/taskiii-dataset/3702_left_knee_1.nii'
mask = load_nifti_image(nii_path)
tibia, femur, background = split_regions(mask)

regions = {
    "tibia": preprocess_region(tibia),
    "femur": preprocess_region(femur),
    "background": preprocess_region(background)
}

In [14]:

# --- Model ---
model_3d = get_3d_densenet().to(device)
model_3d.eval()

DenseNet(
  (features): Sequential(
    (conv0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (norm0): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=(1, 1, 1), ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv3d(64, 128, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
        (norm2): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv3d(128, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm3d(96, ep

In [15]:

# --- Select Conv Layers for Features ---
layer_names = [
    'features.denseblock4.denselayer16.conv2',  # Last
    'features.denseblock4.denselayer12.conv2',  # 3rd-last
    'features.denseblock4.denselayer8.conv2'    # 5th-last
]
extractor = FeatureExtractor3D(model_3d, layer_names).to(device)


In [16]:

# --- Extract Features for Each Region ---
feature_vectors = {}
for name, region_tensor in regions.items():
    with torch.no_grad():
        feats = extractor(region_tensor)
        feature_vectors[name] = [gap_3d(feats[l]) for l in layer_names]


In [17]:
# --- Compute Cosine Similarities ---
pairs = [
    ('tibia', 'femur'),
    ('tibia', 'background'),
    ('femur', 'background')
]
results = []
row = {"image": os.path.basename(nii_path)}
for i, l in enumerate(layer_names):
    for r1, r2 in pairs:
        key = f"{r1}_vs_{r2}_layer{i+1}"
        cos_sim = compute_cosine(feature_vectors[r1][i], feature_vectors[r2][i])
        row[key] = cos_sim
results.append(row)

df = pd.DataFrame(results)
df.to_csv("df.to_csv("/kaggle/working/knee_region_cosine_similarity.csv", index=False)
", index=False)
print(df)


                  image  tibia_vs_femur_layer1  tibia_vs_background_layer1  \
0  3702_left_knee_1.nii               0.999373                    0.999553   

   femur_vs_background_layer1  tibia_vs_femur_layer2  \
0                    0.999933               0.999634   

   tibia_vs_background_layer2  femur_vs_background_layer2  \
0                    0.999799                    0.999937   

   tibia_vs_femur_layer3  tibia_vs_background_layer3  \
0               0.999687                    0.999844   

   femur_vs_background_layer3  
0                    0.999916  
